# Agregar columna source_system a tablas Silver

Este notebook lee la tabla de control `bronze_to_silver_control` y valida que todas las tablas Silver (target_layer.target_schema.target_table) tengan la columna `source_system`. Si no la tienen, la agrega.

**Requisitos:**
- Lakehouse de control adjunto (donde está bronze_to_silver_control)
- Lakehouse Silver adjunto (donde están las tablas target)
- Permisos de escritura en las tablas Silver

## 1. Configuración

In [ ]:
# Lakehouse de control (donde está bronze_to_silver_control)
CONTROL_LAKEHOUSE = "lh_control_erp"
CONTROL_SCHEMA = "dbo"
CONTROL_TABLE = "bronze_to_silver_control"

# Nombre completo de la tabla de control
CONTROL_TABLE_FULL = f"{CONTROL_LAKEHOUSE}.{CONTROL_SCHEMA}.{CONTROL_TABLE}"

## 2. Leer tabla de control y obtener tablas Silver target

In [ ]:
# Leer tabla de control
df_control = spark.sql(f"""
    SELECT DISTINCT
        COALESCE(NULLIF(TRIM(target_layer), ''), 'lh_silver_erp') AS target_layer,
        COALESCE(NULLIF(TRIM(target_schema), ''), 'dbo') AS target_schema,
        TRIM(target_table) AS target_table
    FROM {CONTROL_TABLE_FULL}
    WHERE target_table IS NOT NULL
      AND TRIM(target_table) <> ''
    ORDER BY target_layer, target_schema, target_table
""")

targets = df_control.collect()
print(f"Tablas Silver a validar: {len(targets)}")

In [ ]:
# Mostrar las tablas a procesar
df_control.show(truncate=False)

## 3. Validar y agregar columna source_system

In [ ]:
from pyspark.sql.utils import AnalysisException

COLUMN_TO_ADD = "source_system"
COLUMN_TYPE = "STRING"  # Compatible con Delta/Fabric

added = []
skipped_has_column = []
skipped_not_found = []
errors = []

for row in targets:
    layer = row["target_layer"] or "lh_silver_erp"
    schema = row["target_schema"] or "dbo"
    table = row["target_table"]
    full_name = f"{layer}.{schema}.{table}"
    
    try:
        df = spark.table(full_name)
        columns = [c.lower() for c in df.columns]
        
        if COLUMN_TO_ADD.lower() in columns:
            skipped_has_column.append(full_name)
        else:
            spark.sql(f"ALTER TABLE {full_name} ADD COLUMNS ({COLUMN_TO_ADD} {COLUMN_TYPE})")
            added.append(full_name)
            print(f"✓ Agregada {COLUMN_TO_ADD} en {full_name}")
    
    except AnalysisException as e:
        if "TABLE_OR_VIEW_NOT_FOUND" in str(e) or "does not exist" in str(e).lower():
            skipped_not_found.append(full_name)
        else:
            errors.append((full_name, str(e)))
            print(f"✗ Error en {full_name}: {e}")
    except Exception as e:
        errors.append((full_name, str(e)))
        print(f"✗ Error en {full_name}: {e}")

print(f"\n--- Resumen ---")
print(f"Columnas agregadas: {len(added)}")
print(f"Ya tenían la columna: {len(skipped_has_column)}")
print(f"Tabla no encontrada: {len(skipped_not_found)}")
print(f"Errores: {len(errors)}")

## 4. Detalle de tablas omitidas y errores

In [ ]:
if skipped_not_found:
    print("Tablas no encontradas (no se modificaron):")
    for t in skipped_not_found:
        print(f"  - {t}")

if errors:
    print("\nErrores:")
    for t, msg in errors:
        print(f"  - {t}: {msg}")

## 5. Incluir también bronze_to_silver_control_sap (opcional)

Si deseas procesar las tablas registradas en `bronze_to_silver_control_sap`, descomenta y ejecuta esta celda en lugar de la celda 2.

In [ ]:
# Descomentar para incluir bronze_to_silver_control_sap
# df_control_qad = spark.sql(f"""
#     SELECT DISTINCT COALESCE(NULLIF(TRIM(target_layer),''),'lh_silver_erp') AS target_layer,
#            COALESCE(NULLIF(TRIM(target_schema),''),'dbo') AS target_schema,
#            TRIM(target_table) AS target_table
#     FROM lh_control_erp.dbo.bronze_to_silver_control
#     WHERE target_table IS NOT NULL AND TRIM(target_table)<>''
# """)
# df_control_sap = spark.sql(f"""
#     SELECT DISTINCT COALESCE(NULLIF(TRIM(target_layer),''),'lh_silver_erp') AS target_layer,
#            COALESCE(NULLIF(TRIM(target_schema),''),'dbo') AS target_schema,
#            TRIM(target_table) AS target_table
#     FROM lh_control_erp.dbo.bronze_to_silver_control_sap
#     WHERE target_table IS NOT NULL AND TRIM(target_table)<>''
# """)
# df_control = df_control_qad.union(df_control_sap).distinct().orderBy("target_layer", "target_schema", "target_table")
# targets = df_control.collect()
# print(f"Tablas Silver a validar (QAD + SAP): {len(targets)}")